In [1]:
from collections import Counter
import re

def find_words(text): return re.findall(r'\w+', text.lower())

VOCAB = Counter(find_words(open("big.txt").read()))
INDIR_BIGRAM = {}
DIR_BIGRAM = {}

TOP_K = 20

In [3]:
# Read first file
with open("data/bigrams.txt", "w") as f:
	data = f.readlines()
#
# for line in data:
# 	occur, w1, w2 = line.split()
# 	occur = int(occur)
# 	w1, w2 = w1.lower(), w2.lower()
#
# 	if w1 not in DIR_BIGRAM.keys():
# 		DIR_BIGRAM[w1] = {}
#
# 	if w2 not in INDIR_BIGRAM.keys():
# 		INDIR_BIGRAM[w2] = {}
#
# 	DIR_BIGRAM[w1][w2] = occur
# 	INDIR_BIGRAM[w2][w1] = occur




# Read second file
with open("data/coca_all_links.txt", "w") as f:
	data = f.readlines()


for line in data:
	line = line.split()
	occur, w1, w2 = int(line[0]), line[1].lower(), line[2].lower()


	if w1 not in DIR_BIGRAM.keys():
		DIR_BIGRAM[w1] = []

	if w2 not in DIR_BIGRAM[w1].keys():
		DIR_BIGRAM[w1][w2] = 0

	DIR_BIGRAM[w1][w2] += occur



	if w2 not in INDIR_BIGRAM.keys():
		INDIR_BIGRAM[w2] = {}

	if w1 not in INDIR_BIGRAM[w2].keys():
		INDIR_BIGRAM[w2][w1] = 0

	INDIR_BIGRAM[w2][w1] += occur


UnsupportedOperation: not readable

In [ ]:
def calc_probs(vocab):
	for key in vocab.keys():
		words = vocab[key].items()
		total = sum([num for _, num in words])
		vocab[key] = map(lambda x: (x[0], x[1]/total), words.items())

calc_probs(DIR_BIGRAM)
calc_probs(INDIR_BIGRAM)

In [ ]:
def P(word):
	return VOCAB[word] / sum(VOCAB.values())


def is_known(word):
	return [w for w in word if w in VOCAB]


def edit1(word):
    letters    = 'abcdefghijklmnopqrstuvwxyz'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)


def edit2(word, eds=None):
	if eds is None:
		eds = edit1(word)
	return (e2 for e1 in eds for e2 in edit1(e1))


def norvig_solution(word):
	eds1 = edit1(word)
	eds2 = edit2(word, eds1)
	candidates = is_known([word]) or is_known(eds1) or is_known(eds2)
	# TODO keep more candidates for precision (???)
	return max(candidates, key=P)

In [ ]:
def levenstein_distance(word1, word2):
	pass

In [ ]:
def look_forward(word, next_word):
	prev_words = INDIR_BIGRAM[next_word]

	max_candidates = min(len(INDIR_BIGRAM), TOP_K)

	candidates = prev_words[:max_candidates]
	return candidates

In [ ]:
def look_behind(word, prev_word):
	next_words = DIR_BIGRAM[prev_word]

	max_candidates = min(len(DIR_BIGRAM), TOP_K)

	candidates = next_words[:max_candidates]

	return candidates

In [ ]:
def correction(word, candidates):
	dists = [levenstein_distance(word, c) for c in candidates]
	sorted_candidates = list(zip(candidates, dists)).sort(key=lambda x: x[1], reverse=True)
	return sorted_candidates[0]

In [ ]:
def text_correction(text):
	text = text.split()

	output = []

	for idx, word in enumerate(text):
		candidates = []

		# Base case - Norvig candidates
		norvig_candidates = norvig_solution(word)
		candidates.extend(norvig_candidates)

		# Look forward
		if idx != len(text)-1:
			forward_candidates = look_forward(word, text[idx+1])
			candidates.extend(forward_candidates)

		# Look Behind
		if idx != 0:
			behind_candidates = look_behind(word, text[idx-1])
			candidates.extend(behind_candidates)

		# TODO consider levenstein distance
		corrected = correction(word, candidates)
		output.append(corrected)

	return " ".join(output)

In [1]:
text = "Hello World this is my final message"
output = text_correction(text)
print("Corrected text:\n\t")
print(output)

In [ ]:
# Your code here